# 04 — Agentic AI, No LangChain (manual ReAct loop)
Build the agent loop by hand: the model proposes an action, you parse and execute it, feed the result back, repeat. This is what every agent framework is automating under the hood — see it once manually and frameworks stop feeling like magic.

# Setup
Run this first in every notebook. It assumes this notebook lives in a folder
that can reach `inhouse_wrappers.py` (the CORRECTED version, from `wrapper_fix/`),
`rag_pure_python.py`, and `inhouse_llm.py`. Adjust the `sys.path.append(...)`
lines below if your folder layout differs.

**Corrected in this version:** uses `ask()`/`ask_vision()` (built on the fixed
`get_chat_model()`) instead of calling `multimodal_chat()` directly — the
original always hit the Qwen3-14B endpoint regardless of which `model=` you
asked for. Embeddings go through `embedder.embed_query()`/`.embed_documents()`
instead of `get_embedding(text, model=MODEL_JINA)`, which doesn't match the
real function signature in your `inhouse_llm.py` (no `model=` kwarg there).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../wrapper_fix"))           # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("."))                          # folder containing inhouse_llm.py / rag_pure_python.py
# sys.path.append("/path/to/inhouse_rag_capstone")              # uncomment & adjust if needed

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Define tools as plain Python functions

In [ ]:
def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

def knowledge_lookup(query: str) -> str:
    facts = [
        "MCP standardizes how LLMs call external tools through a client-server interface.",
        "RAG combines a retriever and a generator to ground LLM answers in retrieved context.",
        "An agent is an LLM that chooses actions in a loop based on observations.",
    ]
    store = SimpleVectorStore()
    store.add(facts)
    top = store.search(query, k=1)[0][0]
    return top

TOOLS = {"calculator": calculator, "knowledge_lookup": knowledge_lookup}

## 2. The ReAct prompt contract
**Why this format:** the model needs an unambiguous way to say 'call a tool' vs 'I'm done'. JSON-per-step is simple to parse reliably with a small model.

**Corrected:** `agent_step` now calls `ask()` instead of `multimodal_chat()` directly — `model=MODEL_QWEN3_30B` now genuinely hits the Qwen3-30B endpoint.

In [ ]:
REACT_SYSTEM = """You are an agent with access to these tools:
- calculator(expression): evaluates a math expression
- knowledge_lookup(query): looks up a fact

At each step, respond with ONLY one JSON object, no extra text, in one of two forms:
{"action": "<tool_name>", "action_input": "<input>"}
{"action": "final_answer", "action_input": "<your answer to the user>"}
"""

def agent_step(history_text):
    return ask(REACT_SYSTEM, history_text, model=MODEL_QWEN3_30B, max_tokens=200)

## 3. The loop

In [ ]:
import json

def run_agent(question, max_steps=5):
    transcript = f"Question: {question}"
    for step in range(max_steps):
        raw = agent_step(transcript)
        print(f"--- Step {step+1} raw model output ---\n{raw}\n")
        try:
            action = json.loads(raw)
        except json.JSONDecodeError:
            print("Could not parse action JSON — stopping.")
            return None

        if action["action"] == "final_answer":
            return action["action_input"]

        tool_fn = TOOLS.get(action["action"])
        if tool_fn is None:
            observation = f"Unknown tool: {action['action']}"
        else:
            observation = tool_fn(action["action_input"])

        transcript += f"\nAction: {action}\nObservation: {observation}"
    return "Max steps reached without a final answer."

answer = run_agent("What is MCP, and what is 12 * 7?")
print("FINAL ANSWER:", answer)

### Why practice this manually
Watch the raw transcript output. Notice failure modes you'll see constantly: the model forgetting to call a needed tool, malformed JSON, or looping. Debugging an agent always starts here — at the raw transcript — regardless of which framework you use later.